<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/LanceDB_vectorDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vectors, the output data format of Neural Network models, can effectively encode information and serve a pivotal role in AI applications such as knowledge base, semantic search, Retrieval Augmented Generation (RAG) and more.

LanceDB is an open-source, serverless vector database that is embedded directly into your application. In this guide, we will walk you through how to set up LanceDB locally within minutes and use the Python client library to store and search vectors.

## Install Dependencies
First, we will install the required libraries: `lancedb`, `sentence-transformers`, `pandas`, and `pydantic`.

In [1]:
!pip install lancedb sentence-transformers pandas pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 16.6 MB/s eta 0:00:00


## Set Up Database and Embedding Model
To create a local LanceDB vector database, simply connect to a local directory (like a .db file but for folders). We will also define our embedding model using the `sentence-transformers` registry within LanceDB.

In [2]:
import lancedb
import pandas as pd
import pyarrow as pa
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
import os
from google.colab import userdata



In [3]:
# Connects to a local directory (like a .db file but for folders)
db = lancedb.connect("./lancedb_is_research")

# Select the embedding model
registry = get_registry().get("sentence-transformers")
model = registry.create(name="all-MiniLM-L6-v2")

os.environ["HF_TOKEN"] = userdata.get('HUGGINGFACE_TOKEN')

## Define the Schema
In LanceDB, we define the schema to specify how the data should be structured and embedded. Here, we tell LanceDB to automatically embed the 'Abstract' column.

In [4]:
class Papers(LanceModel):
    id: int
    Year: int
    Title: str
    Abstract: str = model.SourceField()
    URL: str
    JournalFN: str
    # The vector field is automatically populated by the model above
    vector: Vector(model.ndims()) = model.VectorField()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Prepare and Load Data
Load our existing dataset, `ISResearch.csv`, into a Pandas DataFrame. Then, create the table and add the data. LanceDB handles the embedding generation automatically during the insertion process.

In [5]:
# Load your CSV (ensure ISResearch.csv is in your directory)
df = pd.read_csv("https://raw.githubusercontent.com/KarAnalytics/datasets/refs/heads/master/ISResearch.csv")

df.head()

,id,Year,Title,Abstract,URL,JournalFN
0,1,2024,Digital Approaches to Societal Grand Challenge...,Information systems (IS) scholars have pursued...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
1,2,2024,Mr. Right or Mr. Best: The Role of Information...,This paper examines the role of information in...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
2,3,2024,How Information Technology Overcomes Deficienc...,Innovation is vital for the growth of small an...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
3,4,2024,Strategic Expectation Setting of Delivery Time...,Delivery speed is an essential component of th...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research
4,5,2024,User-Generated Content Shapes Judicial Reasoni...,Legal professionals have access to many differ...,https://pubsonline.informs.org/doi/abs/10.1287...,Information Systems Research


In [6]:
# Create the table and add the data
# LanceDB handles the embedding generation during the 'add' process
table = db.create_table("ISResearch", schema=Papers, mode="overwrite")
table.add(df)

print(f"Migration Complete. Total papers in DB: {len(table)}")

Migration Complete. Total papers in DB: 227


## Semantic Search
Now we can perform semantic searches. LanceDB seamlessly embeds the query automatically and returns a Pandas DataFrame for easy viewing.

In [7]:
query = "Which papers mention blockchain or decentralized systems?"

# We search using raw text; LanceDB embeds the query automatically
results = table.search(query).limit(5).to_pandas()

print("\n--- Top 5 Semantic Search Results ---")
print(results[['Title', 'Year', '_distance']])

# Optional: To show the URL of the top result
print(f"\nTop Match URL: {results.iloc[0]['URL']}")


--- Top 5 Semantic Search Results ---
                                               Title  Year  _distance
0  Skin in the Game: The Transformational Potenti...  2024   1.116057
1  Foundations of Decentralized Metaverse Economi...  2025   1.227326
2  Digitization of Transaction Terms within TCE: ...  2024   1.281895
3  Digital Transformation of Academic Publishing:...  2024   1.346166
4  Uncovering the Structural Assurance Mechanisms...  2025   1.373132

Top Match URL: https://misq.umn.edu/skin-the-the-game-the-transformational-potential-of-decentralized-autonomous-organizations.html
